# Six Men's Morris -- AI Agent

**Group 21 -- CS814**

An AI agent that plays Six Men's Morris using minimax search with
alpha-beta pruning and Monte Carlo Tree Search (MCTS), including a
heuristic board evaluation function and both human-vs-AI and AI-vs-AI
game loops.

### Bugs fixed in this revision
- **`MILLS` included 6 invalid, non-collinear patterns** (e.g. `[1, 4, 11]`,
  where 4 and 11 aren't even adjacent on the board). These caused false
  mill detections. Corrected to the 8 real 3-in-a-row lines implied by
  the board's own adjacency graph.
- **`is_terminal()` / `get_winner()` called `self.get_valid_actions()`
  as a method**, but it's a module-level function -- this raised an
  `AttributeError` the moment any game reached the movement phase
  (i.e. almost every real game).
- **MCTS backpropagation credited every node's wins to a single fixed
  player** instead of the player who actually moved into that node,
  which broke UCT's minimax alternation and made the search behave
  closer to random rollouts than genuine tree search.

### Upgrades added
- Standard **flying rule**: a player reduced to 3 pieces may move to
  any empty point, not just an adjacent one.
- **Move ordering** in alpha-beta (mill-forming actions tried first)
  for better pruning.
- A **move-count safety cap** on human-vs-AI games, matching the one
  AI-vs-AI already had.

All fixes were verified by running full AI-vs-AI games end to end
(confirming the movement phase no longer crashes and games reach a
real decisive winner) before being applied here.


In [ ]:
###Group 21###
###Subject Code: CS814
### Six Men's Morris -- AI Agent (Alpha-Beta + MCTS)

import math
import random
from typing import List, Tuple, Optional

# ============================================================================
# TASK 1: STATE REPRESENTATION
# ============================================================================

class GameState:
    """
    Represents a Six Men's Morris game state.

    State encoding:
    - board: List of 16 positions (0-15) representing the board
             0 = empty, 1 = player 1, 2 = player 2
    - phase: 'placement' or 'movement'
    - pieces_to_place: dict {1: count, 2: count} for pieces remaining to place
    - current_player: 1 or 2
    - pieces_on_board: dict {1: count, 2: count}

    Board layout (positions 0-15):

         0 -------- 1 -------- 2
         |          |          |
         |    3 --- 4 --- 5    |
         |    |           |    |
         6 -- 7           8 -- 9
         |    |           |    |
         |    10 -- 11 -- 12   |
         |          |          |
         13 ------- 14 ------- 15
    """

    ADJACENCY = {
        0: [1, 6],
        1: [0, 2, 4],
        2: [1, 9],
        3: [4, 7],
        4: [1, 3, 5],
        5: [4, 8],
        6: [0, 7, 13],
        7: [3, 6, 10],
        8: [5, 9, 12],
        9: [2, 8, 15],
        10: [7, 11],
        11: [10, 12, 14],
        12: [8, 11],
        13: [6, 14],
        14: [11, 13, 15],
        15: [9, 14]
    }

    # Only 8 real mills exist on this board: each of the two concentric
    # squares contributes 4 edges. The 4 "spoke" segments that connect the
    # squares (1-4, 11-14, 6-7, 8-9) are only 2 cells long, so they can
    # never hold 3 collinear pieces and never form a mill on their own.
    # (The previous version listed 14 "mills", 6 of which weren't even
    # collinear -- e.g. [1, 4, 11], where 4 and 11 aren't adjacent at all.)
    MILLS = [
        [0, 1, 2], [3, 4, 5], [10, 11, 12], [13, 14, 15],
        [0, 6, 13], [3, 7, 10], [5, 8, 12], [2, 9, 15],
    ]

    def __init__(self):
        self.board = [0] * 16
        self.phase = 'placement'
        self.pieces_to_place = {1: 6, 2: 6}
        self.current_player = 1
        self.pieces_on_board = {1: 0, 2: 0}

    def copy(self):
        """Create a deep copy of the state"""
        new_state = GameState()
        new_state.board = self.board.copy()
        new_state.phase = self.phase
        new_state.pieces_to_place = self.pieces_to_place.copy()
        new_state.current_player = self.current_player
        new_state.pieces_on_board = self.pieces_on_board.copy()
        return new_state

    def is_terminal(self) -> bool:
        """Check if this is a terminal state"""
        if self.phase == 'placement':
            return False

        # A player loses if they have less than 3 pieces or no valid moves
        opponent = 3 - self.current_player

        if self.pieces_on_board[opponent] < 3:
            return True

        if not get_valid_actions(self):
            return True

        return False

    def get_winner(self) -> Optional[int]:
        """Return winner (1 or 2) or None if not terminal or draw"""
        if not self.is_terminal():
            return None

        opponent = 3 - self.current_player

        if self.pieces_on_board[opponent] < 3:
            return self.current_player

        if not get_valid_actions(self):
            return opponent

        return None

    def check_mill(self, position: int, player: int) -> bool:
        """Check if placing/moving to position forms a mill for player"""
        for mill in self.MILLS:
            if position in mill:
                if all(self.board[pos] == player for pos in mill):
                    return True
        return False

    def get_removable_pieces(self, player: int) -> List[int]:
        """Get list of opponent pieces that can be removed"""
        opponent = 3 - player
        removable = []

        for pos in range(16):
            if self.board[pos] == opponent:
                # Can only remove pieces not in mills, unless all pieces are in mills
                if not self.check_mill(pos, opponent):
                    removable.append(pos)

        # If all opponent pieces are in mills, can remove any
        if not removable:
            removable = [pos for pos in range(16) if self.board[pos] == opponent]

        return removable

# ============================================================================
# TASK 2: ACTIONS AND VALIDATION
# ============================================================================

def get_valid_actions(state: GameState) -> List[Tuple]:
    """
    Get all valid actions for current state.

    Action format:
    - Placement phase: ('place', position, remove_position or None)
    - Movement phase: ('move', from_position, to_position, remove_position or None)
    """
    actions = []

    if state.phase == 'placement':
        # Find empty positions to place pieces
        for pos in range(16):
            if state.board[pos] == 0:
                # Check if placing here forms a mill
                temp_state = state.copy()
                temp_state.board[pos] = state.current_player

                if temp_state.check_mill(pos, state.current_player):
                    # If mill formed, must remove opponent piece
                    removable = temp_state.get_removable_pieces(state.current_player)
                    for remove_pos in removable:
                        actions.append(('place', pos, remove_pos))
                else:
                    actions.append(('place', pos, None))

    else:  # movement phase
        player = state.current_player
        player_positions = [pos for pos in range(16) if state.board[pos] == player]
        empty_positions = [pos for pos in range(16) if state.board[pos] == 0]

        # Flying rule: once a player is down to 3 pieces, they may move to
        # any empty point on the board, not just an adjacent one.
        can_fly = state.pieces_on_board[player] == 3

        for from_pos in player_positions:
            destinations = (
                empty_positions if can_fly
                else [pos for pos in GameState.ADJACENCY[from_pos] if state.board[pos] == 0]
            )

            for to_pos in destinations:
                # Check if moving forms a mill
                temp_state = state.copy()
                temp_state.board[from_pos] = 0
                temp_state.board[to_pos] = player

                if temp_state.check_mill(to_pos, player):
                    removable = temp_state.get_removable_pieces(player)
                    for remove_pos in removable:
                        actions.append(('move', from_pos, to_pos, remove_pos))
                else:
                    actions.append(('move', from_pos, to_pos, None))

    return actions

def apply_action(state: GameState, action: Tuple) -> GameState:
    """Apply an action to a state and return new state"""
    new_state = state.copy()

    if action[0] == 'place':
        _, pos, remove_pos = action
        new_state.board[pos] = state.current_player
        new_state.pieces_to_place[state.current_player] -= 1
        new_state.pieces_on_board[state.current_player] += 1

        if remove_pos is not None:
            opponent = 3 - state.current_player
            new_state.board[remove_pos] = 0
            new_state.pieces_on_board[opponent] -= 1

        # Check if placement phase is complete
        if new_state.pieces_to_place[1] == 0 and new_state.pieces_to_place[2] == 0:
            new_state.phase = 'movement'

    else:  # move
        _, from_pos, to_pos, remove_pos = action
        new_state.board[from_pos] = 0
        new_state.board[to_pos] = state.current_player

        if remove_pos is not None:
            opponent = 3 - state.current_player
            new_state.board[remove_pos] = 0
            new_state.pieces_on_board[opponent] -= 1

    # Switch player
    new_state.current_player = 3 - state.current_player
    return new_state

# ============================================================================
# TASK 3: EVALUATION FUNCTION
# ============================================================================

def evaluate_state(state: GameState, player: int) -> float:
    """
    Evaluate a non-terminal state from perspective of player.

    Higher values are better for the player.

    Features considered:
    - Piece count difference
    - Mills formed
    - Potential mills (2 in a row)
    - Mobility (number of valid moves)
    """

    # Terminal state check
    if state.is_terminal():
        winner = state.get_winner()
        if winner == player:
            return 10000
        elif winner is not None:
            return -10000
        return 0

    opponent = 3 - player
    score = 0

    # 1. Piece count advantage (most important in movement phase)
    piece_diff = state.pieces_on_board[player] - state.pieces_on_board[opponent]
    score += piece_diff * 100

    # 2. Count mills
    player_mills = sum(1 for mill in GameState.MILLS
                      if all(state.board[pos] == player for pos in mill))
    opponent_mills = sum(1 for mill in GameState.MILLS
                         if all(state.board[pos] == opponent for pos in mill))
    score += (player_mills - opponent_mills) * 50

    # 3. Count potential mills (2 pieces in line with empty spot)
    player_potential = 0
    opponent_potential = 0

    for mill in GameState.MILLS:
        player_count = sum(1 for pos in mill if state.board[pos] == player)
        opponent_count = sum(1 for pos in mill if state.board[pos] == opponent)
        empty_count = sum(1 for pos in mill if state.board[pos] == 0)

        if player_count == 2 and empty_count == 1:
            player_potential += 1
        if opponent_count == 2 and empty_count == 1:
            opponent_potential += 1

    score += (player_potential - opponent_potential) * 20

    # 4. Mobility (number of possible moves)
    if state.phase == 'movement':
        original_player = state.current_player

        state.current_player = player
        player_moves = len(get_valid_actions(state))

        state.current_player = opponent
        opponent_moves = len(get_valid_actions(state))

        state.current_player = original_player

        score += (player_moves - opponent_moves) * 5

    return score

# ============================================================================
# TASK 4: ALPHA-BETA ALGORITHM
# ============================================================================

def alpha_beta(state: GameState, depth: int, alpha: float, beta: float,
               maximizing_player: bool, player: int) -> Tuple[float, Optional[Tuple]]:
    """
    Alpha-Beta pruning algorithm.

    Returns: (best_value, best_action)
    """

    # Terminal state or depth limit reached
    if depth == 0 or state.is_terminal():
        return evaluate_state(state, player), None

    valid_actions = get_valid_actions(state)

    if not valid_actions:
        return evaluate_state(state, player), None

    # Move ordering: try mill-forming (capturing) actions first. These are
    # usually strong moves, and trying them first improves the odds of an
    # early alpha/beta cutoff.
    valid_actions = sorted(valid_actions, key=lambda a: a[-1] is None)

    best_action = None

    if maximizing_player:
        max_eval = -math.inf

        for action in valid_actions:
            new_state = apply_action(state, action)
            eval_score, _ = alpha_beta(new_state, depth - 1, alpha, beta, False, player)

            if eval_score > max_eval:
                max_eval = eval_score
                best_action = action

            alpha = max(alpha, eval_score)
            if beta <= alpha:
                break  # Beta cutoff

        return max_eval, best_action

    else:
        min_eval = math.inf

        for action in valid_actions:
            new_state = apply_action(state, action)
            eval_score, _ = alpha_beta(new_state, depth - 1, alpha, beta, True, player)

            if eval_score < min_eval:
                min_eval = eval_score
                best_action = action

            beta = min(beta, eval_score)
            if beta <= alpha:
                break  # Alpha cutoff

        return min_eval, best_action

def get_ai_move(state: GameState, depth: int = 4) -> Tuple:
    """Get best move for AI using alpha-beta pruning"""
    _, best_action = alpha_beta(state, depth, -math.inf, math.inf, True, state.current_player)
    return best_action

# ============================================================================
# TASK 5: HUMAN-AI GAMEPLAY
# ============================================================================

def display_board(state: GameState):
    """Display the current board state"""
    board = state.board

    def piece_char(pos):
        if board[pos] == 1:
            return 'X'
        elif board[pos] == 2:
            return 'O'
        return str(pos).rjust(2)

    print("\n")
    print(f"     {piece_char(0)} -------- {piece_char(1)} -------- {piece_char(2)}")
    print(f"     |          |          |")
    print(f"     |    {piece_char(3)} --- {piece_char(4)} --- {piece_char(5)}    |")
    print(f"     |    |           |    |")
    print(f"     {piece_char(6)} -- {piece_char(7)}           {piece_char(8)} -- {piece_char(9)}")
    print(f"     |    |           |    |")
    print(f"     |    {piece_char(10)} -- {piece_char(11)} -- {piece_char(12)}   |")
    print(f"     |          |          |")
    print(f"     {piece_char(13)} ------- {piece_char(14)} ------- {piece_char(15)}")
    print()
    print(f"Phase: {state.phase}")
    print(f"Player X pieces: {state.pieces_on_board[1]} (to place: {state.pieces_to_place[1]})")
    print(f"Player O pieces: {state.pieces_on_board[2]} (to place: {state.pieces_to_place[2]})")
    print(f"Current player: {'X' if state.current_player == 1 else 'O'}")
    print()

def get_human_move(state: GameState) -> Tuple:
    """Get move from human player"""
    valid_actions = get_valid_actions(state)

    if not valid_actions:
        return None

    while True:
        try:
            if state.phase == 'placement':
                print("Enter position to place piece (0-15):")
                pos = int(input())

                # Check if this forms a mill
                temp_state = state.copy()
                temp_state.board[pos] = state.current_player

                if temp_state.check_mill(pos, state.current_player):
                    print("Mill formed! Enter position of opponent piece to remove:")
                    remove_pos = int(input())
                    action = ('place', pos, remove_pos)
                else:
                    action = ('place', pos, None)

            else:  # movement
                print("Enter FROM position (0-15):")
                from_pos = int(input())
                print("Enter TO position (0-15):")
                to_pos = int(input())

                # Check if this forms a mill
                temp_state = state.copy()
                temp_state.board[from_pos] = 0
                temp_state.board[to_pos] = state.current_player

                if temp_state.check_mill(to_pos, state.current_player):
                    print("Mill formed! Enter position of opponent piece to remove:")
                    remove_pos = int(input())
                    action = ('move', from_pos, to_pos, remove_pos)
                else:
                    action = ('move', from_pos, to_pos, None)

            if action in valid_actions:
                return action
            else:
                print("Invalid move! Try again.")
                print(f"Valid actions: {valid_actions[:5]}...")  # Show first 5

        except (ValueError, IndexError):
            print("Invalid input! Try again.")

def play_human_vs_ai(human_player: int = 1, ai_depth: int = 4, max_moves: int = 300):
    """
    Main game loop for human vs AI.

    Args:
        human_player: 1 or 2 (X or O)
        ai_depth: Search depth for AI
        max_moves: Safety cap to prevent an unbounded game (declared a draw
            if reached, same protection play_ai_vs_ai already had)
    """
    state = GameState()
    ai_player = 3 - human_player
    move_count = 0

    print("=" * 50)
    print("Six Men's Morris - Human vs AI")
    print("=" * 50)
    print(f"You are player {'X' if human_player == 1 else 'O'}")
    print(f"AI is player {'X' if ai_player == 1 else 'O'}")
    print()

    while not state.is_terminal() and move_count < max_moves:
        display_board(state)

        if state.current_player == human_player:
            print("Your turn!")
            action = get_human_move(state)
        else:
            print("AI is thinking...")
            action = get_ai_move(state, ai_depth)
            print(f"AI plays: {action}")

        if action is None:
            break

        state = apply_action(state, action)
        move_count += 1

    # Game over
    display_board(state)
    winner = state.get_winner()

    if winner == human_player:
        print("🎉 Congratulations! You win!")
    elif winner == ai_player:
        print("😔 AI wins! Better luck next time.")
    else:
        print("It's a draw!")

# ============================================================================
# TASK 6: MONTE CARLO TREE SEARCH
# ============================================================================

class MCTSNode:
    """Node for Monte Carlo Tree Search"""

    def __init__(self, state: GameState, parent=None, action=None):
        self.state = state
        self.parent = parent
        self.action = action
        self.children = []
        self.visits = 0
        self.wins = 0
        self.untried_actions = get_valid_actions(state)

    def uct_value(self, exploration_weight: float = 1.41) -> float:
        """Calculate UCT (Upper Confidence Bound for Trees) value"""
        if self.visits == 0:
            return math.inf

        exploitation = self.wins / self.visits
        exploration = exploration_weight * math.sqrt(math.log(self.parent.visits) / self.visits)
        return exploitation + exploration

    def best_child(self, exploration_weight: float = 1.41):
        """Select best child using UCT"""
        return max(self.children, key=lambda c: c.uct_value(exploration_weight))

    def expand(self):
        """Expand node by adding a random untried child"""
        action = self.untried_actions.pop(random.randrange(len(self.untried_actions)))
        new_state = apply_action(self.state, action)
        child = MCTSNode(new_state, parent=self, action=action)
        self.children.append(child)
        return child

    def is_fully_expanded(self) -> bool:
        """Check if all actions have been tried"""
        return len(self.untried_actions) == 0

    def is_terminal_node(self) -> bool:
        """Check if this is a terminal state"""
        return self.state.is_terminal()

def mcts_simulate(state: GameState) -> int:
    """Simulate a random playout from state"""
    current_state = state.copy()

    while not current_state.is_terminal():
        actions = get_valid_actions(current_state)
        if not actions:
            break
        action = random.choice(actions)
        current_state = apply_action(current_state, action)

    winner = current_state.get_winner()
    return winner if winner else 0

def mcts_search(state: GameState, iterations: int = 1000) -> Tuple:
    """
    Monte Carlo Tree Search algorithm.

    Returns best action to take from state.
    """
    root = MCTSNode(state)

    for _ in range(iterations):
        node = root

        # Selection: traverse tree using UCT
        while node.is_fully_expanded() and not node.is_terminal_node():
            node = node.best_child()

        # Expansion: add new child if possible
        if not node.is_terminal_node() and not node.is_fully_expanded():
            node = node.expand()

        # Simulation: random playout
        winner = mcts_simulate(node.state)

        # Backpropagation: update statistics. Each node's win rate must be
        # from the perspective of whichever player actually moved into that
        # node (node.state.current_player is whose turn it is NEXT, so the
        # mover is the other player) -- crediting every node's wins to a
        # single fixed player instead breaks UCT's minimax alternation and
        # makes the tree search no better than random rollouts.
        while node is not None:
            node.visits += 1
            mover = 3 - node.state.current_player
            if winner == mover:
                node.wins += 1
            elif winner == 0:
                node.wins += 0.5
            node = node.parent

    # Return action with most visits
    best_child = max(root.children, key=lambda c: c.visits)
    return best_child.action

def play_ai_vs_ai(ai1_type: str = 'alphabeta', ai2_type: str = 'mcts',
                  depth: int = 4, mcts_iterations: int = 1000, verbose: bool = True):
    """
    Play AI vs AI match.

    Args:
        ai1_type: 'alphabeta' or 'mcts'
        ai2_type: 'alphabeta' or 'mcts'
        depth: Depth for alpha-beta
        mcts_iterations: Iterations for MCTS
        verbose: Print board state
    """
    state = GameState()
    move_count = 0

    if verbose:
        print("=" * 50)
        print(f"AI vs AI: {ai1_type.upper()} (X) vs {ai2_type.upper()} (O)")
        print("=" * 50)

    while not state.is_terminal() and move_count < 200:  # Prevent infinite games
        if verbose:
            display_board(state)

        # Get AI move
        if state.current_player == 1:
            if ai1_type == 'alphabeta':
                action = get_ai_move(state, depth)
            else:
                action = mcts_search(state, mcts_iterations)
            if verbose:
                print(f"AI1 ({ai1_type}) plays: {action}")
        else:
            if ai2_type == 'alphabeta':
                action = get_ai_move(state, depth)
            else:
                action = mcts_search(state, mcts_iterations)
            if verbose:
                print(f"AI2 ({ai2_type}) plays: {action}")

        if action is None:
            break

        state = apply_action(state, action)
        move_count += 1

    if verbose:
        display_board(state)

    winner = state.get_winner()

    if verbose:
        if winner == 1:
            print(f"🏆 AI1 ({ai1_type}) wins!")
        elif winner == 2:
            print(f"🏆 AI2 ({ai2_type}) wins!")
        else:
            print("It's a draw!")

    return winner

# ============================================================================
# MAIN FUNCTION
# ============================================================================

if __name__ == "__main__":
    print("Six Men's Morris AI Agent")
    print("=" * 50)
    print()
    print("Select game mode:")
    print("1. Human vs AI")
    print("2. AI vs AI (Alpha-Beta vs MCTS)")
    print("3. Test single position evaluation")

    choice = input("Enter choice (1-3): ")

    if choice == '1':
        print("\nChoose your player:")
        print("1. Play as X (first)")
        print("2. Play as O (second)")
        player_choice = input("Enter choice (1-2): ")

        human_player = 1 if player_choice == '1' else 2

        print("\nChoose AI difficulty (search depth):")
        print("1. Easy (depth 2)")
        print("2. Medium (depth 4)")
        print("3. Hard (depth 6)")
        difficulty = input("Enter choice (1-3): ")

        depth_map = {'1': 2, '2': 4, '3': 6}
        ai_depth = depth_map.get(difficulty, 4)

        play_human_vs_ai(human_player, ai_depth)

    elif choice == '2':
        play_ai_vs_ai('alphabeta', 'mcts', depth=4, mcts_iterations=500, verbose=True)

    elif choice == '3':
        # Test evaluation function
        state = GameState()
        state.board = [1, 0, 2, 0, 1, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0]
        state.pieces_on_board = {1: 2, 2: 2}
        state.phase = 'movement'

        display_board(state)
        print(f"Evaluation for Player 1: {evaluate_state(state, 1)}")
        print(f"Evaluation for Player 2: {evaluate_state(state, 2)}")
